In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nesandipethiaramba/octvedta/Hotel-A-test.csv
/kaggle/input/datasets/nesandipethiaramba/octvedta/train_data_cleaned_V2_ML (2).csv
/kaggle/input/datasets/nesandipethiaramba/octvedta/Hotel-A-validation.csv


In [2]:
train = "/kaggle/input/datasets/nesandipethiaramba/octvedta/train_data_cleaned_V2_ML (2).csv"
test = "/kaggle/input/datasets/nesandipethiaramba/octvedta/Hotel-A-test.csv"
validation = "/kaggle/input/datasets/nesandipethiaramba/octvedta/Hotel-A-validation.csv"



In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- 1. DIRECTORY SETUP ---
WORK_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(WORK_DIR, "saved_models")
EVAL_DIR = os.path.join(WORK_DIR, "evaluation_results")

for folder in [MODEL_DIR, EVAL_DIR]:
    os.makedirs(folder, exist_ok=True)

# --- 2. DATA LOADING & PREPROCESSING ---
train_path = "/kaggle/input/datasets/nesandipethiaramba/octvedta/train_data_cleaned_V2_ML (2).csv"
df = pd.read_csv(train_path)

# Prepare Features and Target
X_raw = df.drop(columns=['Reservation_Status'])
y_raw = df['Reservation_Status']

# Encoding Features (One-Hot)
X_enc = pd.get_dummies(X_raw, drop_first=True)

# Encoding Target (LabelEncoder)
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)

# Split for internal validation (20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# --- 3. EVALUATION HELPER FUNCTION ---
def evaluate_and_save(model, X, y, model_name, save_dir, color_map='Blues'):
    """Generates and saves all required evaluation metrics."""
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)
    
    # 1. Classification Report
    report = classification_report(y, y_pred, target_names=le.classes_)
    with open(os.path.join(save_dir, f'{model_name}_report.txt'), 'w') as f:
        f.write(report)
        
    # 2. Confusion Matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color_map, 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{model_name}_cm.png'))
    plt.close()
    
    # 3. ROC Curves
    y_bin = label_binarize(y, classes=range(len(le.classes_)))
    plt.figure(figsize=(10, 8))
    colors = ['darkorange', 'cornflowerblue', 'green']
    for i in range(len(le.classes_)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, color=colors[i], lw=2, 
                 label=f'{le.classes_[i]} (AUC = {auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.title(f'ROC Curves: {model_name}')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(save_dir, f'{model_name}_roc.png'))
    plt.close()

# --- 4. MODEL 1: DECISION TREE (SMOTE + TUNED) ---
print("Training Model 1: Decision Tree with SMOTE...")
dt_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42))
])

dt_params = {
    'dt__max_depth': [10, 15, 20],
    'dt__min_samples_leaf': [5, 10],
    'dt__criterion': ['gini', 'entropy']
}

dt_search = RandomizedSearchCV(dt_pipe, dt_params, n_iter=10, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
dt_search.fit(X_train, y_train)
best_dt = dt_search.best_estimator_

# Save DT
joblib.dump(best_dt, os.path.join(MODEL_DIR, 'decision_tree_model.pkl'))
evaluate_and_save(best_dt, X_test, y_test, "DecisionTree", EVAL_DIR, 'Blues')

# --- 5. MODEL 2: GRADIENT BOOSTING (SMOTE + AGGRESSIVE) ---
print("Training Model 2: Gradient Boosting with SMOTE...")
gb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
])

gb_params = {
    'gb__n_estimators': [300, 500],
    'gb__learning_rate': [0.05, 0.1],
    'gb__max_depth': [4, 6],
    'gb__subsample': [0.8, 0.9]
}

gb_search = RandomizedSearchCV(gb_pipe, gb_params, n_iter=10, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
gb_search.fit(X_train, y_train)
best_gb = gb_search.best_estimator_

# Save GB
joblib.dump(best_gb, os.path.join(MODEL_DIR, 'gradient_boosting_model.pkl'))
evaluate_and_save(best_gb, X_test, y_test, "GradBoost", EVAL_DIR, 'Greens')

# Save Label Encoder
joblib.dump(le, os.path.join(MODEL_DIR, 'label_encoder.pkl'))

print("\nSuccess! Check /kaggle/working/ for organized outputs.")

Training Model 1: Decision Tree with SMOTE...
Training Model 2: Gradient Boosting with SMOTE...

Success! Check /kaggle/working/ for organized outputs.
